# Orchestrator-Subagent — LangGraph

**Pattern:** Orchestrator node manages task queue → routes to specialists

```
START → orchestrator ─┬─→ highlights → orchestrator
                      ├─→ logistics  → orchestrator
                      ├─→ itinerary  → orchestrator
                      ├─→ format     → orchestrator
                      └─→ END
```

The orchestrator node pops from a `pending_tasks` list and routes accordingly.
This makes the orchestration logic visible in the graph.

In [ ]:
import os
from typing import TypedDict, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
print("✓ Ready")

In [ ]:
HIGHLIGHTS = {"tokyo": "Shibuya, Senso-ji, Tsukiji", "paris": "Eiffel, Louvre", "bangalore": "Lalbagh, Nandi Hills"}
LOGISTICS  = {"tokyo": "3h from SFO. Shinkansen.", "paris": "11h from NYC. Metro.", "bangalore": "9h from Dubai. Ola."}

class OState(TypedDict):
    city: str; pending_tasks: list; next_task: str
    highlights: Optional[str]; logistics: Optional[str]
    itinerary: Optional[str]; final_package: Optional[str]

print("State defined with pending_tasks queue")

In [ ]:
def orchestrator(s):
    if s["pending_tasks"]:
        nxt = s["pending_tasks"][0]
        print(f"  [orch] → {nxt}")
        return {"pending_tasks": s["pending_tasks"][1:], "next_task": nxt}
    return {"next_task": "done"}

def highlights_node(s):
    r = llm.invoke([SystemMessage(content="List top 3 attractions."), HumanMessage(content=f"{s['city']}: {HIGHLIGHTS.get(s['city'].lower(),'N/A')}")])
    return {"highlights": r.content}

def logistics_node(s):
    r = llm.invoke([SystemMessage(content="Practical travel tips: flights, transport, season."), HumanMessage(content=f"{s['city']}: {LOGISTICS.get(s['city'].lower(),'N/A')}")])
    return {"logistics": r.content}

def itinerary_node(s):
    r = llm.invoke([SystemMessage(content="Create a 3-day itinerary."), HumanMessage(content=f"City: {s['city']}\nHighlights: {s['highlights']}\nLogistics: {s['logistics']}")])
    return {"itinerary": r.content}

def format_node(s):
    r = llm.invoke([SystemMessage(content="Format as polished trip package with ## sections."), HumanMessage(content=f"City: {s['city']}\nHighlights: {s['highlights']}\nLogistics: {s['logistics']}\nItinerary: {s['itinerary']}")])
    return {"final_package": r.content}

print("5 nodes: orchestrator + 4 specialists")

In [ ]:
builder = StateGraph(OState)
for name, fn in [("orchestrator",orchestrator),("highlights",highlights_node),("logistics",logistics_node),("itinerary",itinerary_node),("format",format_node)]:
    builder.add_node(name, fn)
builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", lambda s: s["next_task"],
    {"highlights":"highlights","logistics":"logistics","itinerary":"itinerary","format":"format","done":END})
for n in ["highlights","logistics","itinerary","format"]: builder.add_edge(n, "orchestrator")
graph = builder.compile()
try:
    from IPython.display import Image, display; display(Image(graph.get_graph().draw_mermaid_png()))
except: print(graph.get_graph().draw_mermaid())

In [ ]:
result = graph.invoke({"city": "Tokyo", "pending_tasks": ["highlights","logistics","itinerary","format"],
    "next_task": "", "highlights": None, "logistics": None, "itinerary": None, "final_package": None})
print(result["final_package"])

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `pending_tasks` as a state queue | Orchestrator reads a list → routes one task at a time |
| Workers return to orchestrator | Same hub-and-spoke as Hierarchical, but task-driven |
| Order is data, not code | Changing `pending_tasks` changes the orchestration plan |

**Difference from Hierarchical:** In Hierarchical, the supervisor reads state conditions. Here, the orchestrator pops a task queue — the plan is explicit in state.

### Next: [CrewAI Orchestrator →](../CrewAI/orchestrator.ipynb)